In [1]:
from helpers import init, preprocess_data

In [2]:
SEED = 42
X_train_raw, y_train_cm, X_val_raw, y_val_cm = init(SEED)
X_train, y_train, X_val, y_val = preprocess_data(X_train_raw, y_train_cm, X_val_raw, y_val_cm)

PyTorch: 2.11.0+cu130
Training rows: 42166
Validation rows: 4685
Input shape: (42166, 9)
Target x range (cm): 0.0 to 281.0
Target y range (cm): 0.0 to 275.0
Using split files:
  train: train_clean_3x3_1cm.csv 42166 rows
  validation: validation_clean_3x3_1cm.csv 4685 rows
RSS scale: 0.8493868112564087
Target min cm: [0. 0.]
Target range cm: [281. 275.]


# Get the model

In [3]:
from vlp_hackathon.baseline_model import BaselineMLP
from vlp_hackathon.improved_model import ImprovedMLP
model_cls = ImprovedMLP

# Run a trail

In [9]:
from torch import nn
import torch
import numpy as np
from matplotlib import pyplot as plt

def train_model(model, optimizer, loss_fn, epochs, batch_size, plot: bool = False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)
    model = model.to(device)

    history = []
    train_rng = np.random.default_rng(SEED)
    for epoch in range(epochs):
        model.train()
        permutation = train_rng.permutation(len(X_train))
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            idx = permutation[start:start + batch_size]
            xb = X_train_t[idx]
            yb = y_train_t[idx]
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += float(loss.item()) * len(idx)

        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(X_val_t), y_val_t).item())
        history.append((running / len(X_train), val_loss))
        #print(f"epoch={epoch + 1:02d} train_mse={history[-1][0]:.6f} val_mse={val_loss:.6f}")

    history = np.asarray(history, dtype=np.float32)
    if plot:
        plt.figure(figsize=(6, 4))
        plt.plot(history[:, 0], label="train")
        plt.plot(history[:, 1], label="validation")
        plt.xlabel("Epoch")
        plt.ylabel("MSE on normalized coordinates")
        plt.legend()
        plt.title("Baseline training curve")
        plt.show()

    return np.min(history[:, 1])

def run_trial(trial):
    epochs = 50

    print(trial)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    batch_size = 2048 #trial.suggest_int("batch_size", 16, 2048)

    depth = trial.suggest_int("depth", 1, 6)
    layers = []
    for layer in range(depth):
        layers.append(trial.suggest_int(f"layer_{layer}", 2, 64))

    # Start and end are fixed
    layers = [9] + layers + [2]

    model = model_cls(layers=layers)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    return train_model(model, optimizer, loss_fn, epochs, batch_size)



In [13]:
import optuna
from optuna.trial import TrialState

study = optuna.create_study(direction='minimize', study_name="task_1", storage="sqlite:///optuna/trail_1.db")
study.optimize(run_trial, n_trials=5, show_progress_bar=True)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [12]:
from plotly.io import show

fig = optuna.visualization.plot_optimization_history(study)
show(fig)

fig = optuna.visualization.plot_param_importances(study)
show(fig)

fig = optuna.visualization.plot_slice(study)
show(fig)